In [80]:
import json
import pandas as pd
import numpy as np
import altair as alt
from scipy import stats

### Setup (Loading data, minor preprocesing)

In [98]:
with open('dataset.json', 'r') as f:
    raw = json.load(f)

RESOLUTIONS = ['50', '71', '87', '100']
PARAMS = ['alpha_weight', 'hist_percent', 'filter_size', 'num_samples']

records = []

for scene, scene_data in raw.items():
    if not isinstance(scene_data, dict):
        continue
    for res in RESOLUTIONS:
        if res not in scene_data:
            continue
        for ref, ref_data in scene_data[res].items():
            if not ref.startswith('ref-'):
                continue
            if ref != f'ref-{scene}':
                continue
            for param in PARAMS:
                if param not in ref_data:
                    continue
                for entry in ref_data[param]:
                    records.append({
                        'scene': scene,
                        'resolution': int(res),
                        'parameter': param,
                        'value': entry['value'],
                        'score': entry['score'],
                    })

df = pd.DataFrame(records)

# Find common values per (parameter, resolution) across all scenes
common_values = df.groupby(['parameter', 'resolution', 'value'])['scene'].nunique()
max_scenes = df.groupby(['parameter', 'resolution'])['scene'].nunique()

def get_common_values(group):
    key = (group.name[1], group.name[2])  # (parameter, resolution)
    total_scenes = df[(df['parameter'] == key[0]) & (df['resolution'] == key[1])]['scene'].nunique()
    common = group.groupby('value')['scene'].nunique()
    return common[common == total_scenes].index.tolist()

# Center only over common values
def center_score(group):
    param, res = group['parameter'].iloc[0], group['resolution'].iloc[0]
    common = df[(df['parameter'] == param) & (df['resolution'] == res)]
    common_vals = common.groupby('value')['scene'].nunique()
    n_scenes = common['scene'].nunique()
    shared_vals = common_vals[common_vals == n_scenes].index
    mask = group['value'].isin(shared_vals)
    mean = group.loc[mask, 'score'].mean()
    return group['score'] - mean

df['score_centered'] = df.groupby(['scene', 'parameter', 'resolution'], group_keys=False).apply(center_score)
# Ignore hist percent less than 100 (measurement exists from prev trials but is not significant)
df = df[~((df['parameter'] == 'hist_percent') & (df['value'] < 100))]

# These are the weird outlier scenes--we cut them
EXCLUDE_SCENES = ['junkyard-mound1', 'junkyard-mound2', 'oldmine-speed-18', 'oldmine-speed-35', 'oldmine-speed-75', 'oldmine-speed-9', 'oldmine-warm']
df = df[~df['scene'].isin(EXCLUDE_SCENES)]

# Print for sanity check
print(df.shape)
df.head(10)

(2348, 6)


/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_43257/835408140.py:55: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['score_centered'] = df.groupby(['scene', 'parameter', 'resolution'], group_keys=False).apply(center_score)


,scene,resolution,parameter,value,score,score_centered
0,abandoned,50,alpha_weight,0.01,91.475502,-0.130814
1,abandoned,50,alpha_weight,0.02,91.696266,0.089950
2,abandoned,50,alpha_weight,0.04,92.162430,0.556114
3,abandoned,50,alpha_weight,0.06,92.602554,0.996238
4,abandoned,50,alpha_weight,0.10,93.051781,1.445464
5,abandoned,50,alpha_weight,0.20,93.491249,1.884933
6,abandoned,50,alpha_weight,0.50,92.588005,0.981689
7,abandoned,50,alpha_weight,0.60,91.999229,0.392913
8,abandoned,50,alpha_weight,0.70,91.279709,-0.326607
9,abandoned,50,alpha_weight,0.80,90.529510,-1.076806


In [99]:
df.groupby(['parameter', 'resolution'])['score'].agg(['count', 'mean', 'std']).round(3)

count    mean    std
parameter    resolution                      
alpha_weight 50            252  86.141  8.881
             71            252  91.326  5.985
             87            252  93.199  4.957
             100           252  94.277  4.443
filter_size  50            146  87.444  7.482
             71            147  91.156  6.175
             87            147  92.444  5.636
             100           146  93.339  5.279
hist_percent 50             84  87.392  7.391
             71             84  91.853  5.674
             87             84  93.219  5.144
             100            82  94.015  4.814
num_samples  50            105  87.281  7.673
             71            105  91.172  6.186
             87            105  92.459  5.639
             100           105  93.279  5.299

### Basic Info about Data

In [100]:
print(f"Scenes: {df['scene'].nunique()}")
print(f"Scene names: {df['scene'].unique().tolist()}")
print(f"\nParameter value counts:")
print(df.groupby(['parameter', 'value'])['scene'].count().rename('n_scenes'))

Scenes: 21
Scene names: ['abandoned', 'abandoned-demo', 'abandoned-flipped', 'cubetest', 'fantasticvillage-open', 'lightfoliage', 'lightfoliage-close', 'oldmine', 'quarry-all', 'quarry-rocksonly', 'resto-close', 'resto-fwd', 'resto-pan', 'scifi', 'subway-lookdown', 'subway-turn', 'wildwest-bar', 'wildwest-barzoom', 'wildwest-behindcounter', 'wildwest-store', 'wildwest-town']

Parameter value counts:
parameter     value 
alpha_weight  0.01      84
              0.02      84
              0.04      84
              0.06      84
              0.10      84
              0.20      84
              0.50      84
              0.60      84
              0.70      84
              0.80      84
              0.90      84
              1.00      84
filter_size   0.10      84
              0.25      84
              0.50      83
              0.70      83
              0.90      84
              0.95      84
              1.00      84
hist_percent  100.00    84
              125.00    84
         

In [101]:
# Print out for each scene x resolution, whether the param is calculated 
for param in PARAMS:
    sub = df[df['parameter'] == param]
    pivot = sub.groupby(['scene', 'resolution', 'value'])['score'].count().unstack('value')
    print(f"\n=== {param} ===")
    print(pivot.to_string())


=== alpha_weight ===
value                              0.01  0.02  0.04  0.06  0.10  0.20  0.50  0.60  0.70  0.80  0.90  1.00
scene                  resolution                                                                        
abandoned              50             1     1     1     1     1     1     1     1     1     1     1     1
                       71             1     1     1     1     1     1     1     1     1     1     1     1
                       87             1     1     1     1     1     1     1     1     1     1     1     1
                       100            1     1     1     1     1     1     1     1     1     1     1     1
abandoned-demo         50             1     1     1     1     1     1     1     1     1     1     1     1
                       71             1     1     1     1     1     1     1     1     1     1     1     1
                       87             1     1     1     1     1     1     1     1     1     1     1     1
                       1

In [102]:
# Find missing params
for param in PARAMS:
    sub = df[df['parameter'] == param]
    pivot = sub.groupby(['scene', 'resolution', 'value'])['score'].count().unstack('value')
    missing = pivot[pivot.isna().any(axis=1)]
    if not missing.empty:
        print(f"\n=== {param} ===")
        print(missing.to_string())


=== hist_percent ===
value                      100.0  125.0  150.0  200.0
scene          resolution                            
abandoned-demo 100           1.0    1.0    NaN    1.0
cubetest       100           1.0    1.0    1.0    NaN

=== filter_size ===
value                          0.10  0.25  0.50  0.70  0.90  0.95  1.00
scene              resolution                                          
abandoned-flipped  100          1.0   1.0   NaN   1.0   1.0   1.0   1.0
lightfoliage-close 50           1.0   1.0   1.0   NaN   1.0   1.0   1.0


### Statistical Significance of Parameters

#### Anova Test (said everythign was insignificant)

In [103]:
# Anova shows everything as insignificant...
anova_records = []

for (param, res), group in df.groupby(['parameter', 'resolution']):
    groups_by_value = [g['score'].values for _, g in group.groupby('value')]
    if len(groups_by_value) < 2:
        continue
    f_stat, p_val = stats.f_oneway(*groups_by_value)
    anova_records.append({
        'parameter': param,
        'resolution': res,
        'F': round(f_stat, 4),
        'p_value': round(p_val, 6),
        'significant': p_val < 0.05,
    })

anova_df = pd.DataFrame(anova_records)
anova_df

,parameter,resolution,F,p_value,significant
0,alpha_weight,50,0.9945,0.451983,False
1,alpha_weight,71,0.4638,0.924072,False
2,alpha_weight,87,0.5460,0.870472,False
3,alpha_weight,100,0.8543,0.585960,False
4,filter_size,50,0.0502,0.999473,False
5,filter_size,71,0.0000,1.000000,False
6,filter_size,87,0.0000,1.000000,False
7,filter_size,100,0.0029,1.000000,False
8,hist_percent,50,0.0234,0.995114,False
9,hist_percent,71,0.1846,0.906569,False


In [104]:
# Setup plot color & orders (for consistency)
res_order = ['100%', '87%', '71%', '50%']
res_colors = [ '#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']

color_scale = alt.Scale(domain=res_order, range=res_colors)


In [120]:
effect_records = []

for (param, res), group in df.groupby(['parameter', 'resolution']):
    scene_ranges = group.groupby('scene').apply(
        lambda s: s.groupby('value')['score'].mean().max() - s.groupby('value')['score'].mean().min()
    )
    effect_records.append({
        'parameter': param,
        'resolution': res,
        'mean_range': scene_ranges.mean(),
        'se_range': scene_ranges.sem()
    })

effect_df = pd.DataFrame(effect_records)
effect_df['range_label'] = effect_df['mean_range'].apply(lambda x: f'{x:.2f}')

/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_43257/1139419750.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scene_ranges = group.groupby('scene').apply(
/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_43257/1139419750.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scene_ranges = group.groupby('scene').apply(
/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_43257/113941975

In [126]:
from scipy.stats import friedmanchisquare

# ── Friedman ──────────────────────────────────────────────────────────────────
def compute_friedman(df):
    records = []
    for (param, res), group in df.groupby(['parameter', 'resolution']):
        value_counts = group.groupby('scene')['value'].nunique()
        n_values = group['value'].nunique()
        complete_scenes = value_counts[value_counts == n_values].index
        group = group[group['scene'].isin(complete_scenes)]
        if group['scene'].nunique() < 3:
            continue
        values = sorted(group['value'].unique())
        groups_by_value = [group[group['value'] == v]['score_centered'].values for v in values]
        if any(len(g) < 3 for g in groups_by_value):
            continue
        _, p = friedmanchisquare(*groups_by_value)
        records.append({'parameter': param, 'resolution': res, 'p_value': p, 'test': 'Friedman'})
    out = pd.DataFrame(records)
    out['significant'] = out['p_value'] < 0.05
    return out

# ── Kendall's W ───────────────────────────────────────────────────────────────
def kendalls_w(data):
    n, k = data.shape
    ranks = data.rank(axis=1)
    rank_sums = ranks.sum(axis=0)
    S = ((rank_sums - rank_sums.mean()) ** 2).sum()
    W = 12 * S / (n ** 2 * (k ** 3 - k))
    chi2 = n * (k - 1) * W
    p = 1 - stats.chi2.cdf(chi2, df=k - 1)
    return W, p

def compute_kendall(df):
    records = []
    for (param, res), group in df.groupby(['parameter', 'resolution']):
        value_counts = group.groupby('scene')['value'].nunique()
        n_values = group['value'].nunique()
        complete_scenes = value_counts[value_counts == n_values].index
        group = group[group['scene'].isin(complete_scenes)]
        if group['scene'].nunique() < 3:
            continue
        pivot = group.pivot_table(index='scene', columns='value', values='score_centered').dropna()
        if pivot.shape[0] < 3 or pivot.shape[1] < 2:
            continue
        W, p = kendalls_w(pivot)
        records.append({'parameter': param, 'resolution': res, 'p_value': p, 'W': round(W, 4), 'test': 'Kendall W'})
    out = pd.DataFrame(records)
    out['significant'] = out['p_value'] < 0.05
    return out

# ── Spearman ──────────────────────────────────────────────────────────────────
def compute_spearman(df):
    records = []
    for (param, res), group in df.groupby(['parameter', 'resolution']):
        scene_corrs = []
        for scene, sgroup in group.groupby('scene'):
            val_means = sgroup.groupby('value')['score_centered'].mean()
            if len(val_means) < 3:
                continue
            r, _ = stats.spearmanr(val_means.index, val_means.values)
            scene_corrs.append(r)
        if len(scene_corrs) < 3:
            continue
        t, p = stats.ttest_1samp(scene_corrs, 0)
        records.append({'parameter': param, 'resolution': res, 'mean_r': round(np.mean(scene_corrs), 4), 'p_value': round(p, 6), 'test': 'Spearman'})
    out = pd.DataFrame(records)
    out['significant'] = out['p_value'] < 0.05
    return out

In [127]:
# ── Compute all ───────────────────────────────────────────────────────────────
sig_friedman = compute_friedman(df)
sig_kendall  = compute_kendall(df)
sig_spearman = compute_spearman(df)

In [ ]:
# ── Toggle this to switch tests in all downstream plots ───────────────────────
sig_df = sig_kendall   # ← change to sig_friedman, sig_kendall, or sig_spearman
print(sig_df)

In [124]:
agg = df.groupby(['parameter', 'resolution', 'value'])['score'].agg(['mean', 'std', 'count']).reset_index()
agg['ci95'] = 1.96 * (agg['std'] / np.sqrt(agg['count']))
agg['resolution'] = agg['resolution'].astype(str) + '%'

y_min = (agg['mean'] - agg['ci95']).min() - 2  # a little padding
y_max = 100
y_scale = alt.Scale(domain=[y_min, y_max])

cgvqm_labels = pd.DataFrame([
    {'score': 100, 'label': 'Imperceptible'},
    {'score': 75,  'label': 'Perceptible but not annoying'},
    {'score': 50,  'label': 'Slightly annoying'},
    {'score': 25,  'label': 'Annoying'},
    {'score': 0,   'label': 'Very annoying'},
])
cgvqm_labels = cgvqm_labels[cgvqm_labels['score'] >= y_min]

rule = alt.Chart(cgvqm_labels).mark_rule(strokeDash=[4, 4], opacity=0.4, color='gray').encode(
    y=alt.Y('score:Q', scale=y_scale)
)

label = alt.Chart(cgvqm_labels).mark_text(align='left', baseline='middle', dx=4, fontSize=9, color='gray').encode(
    y=alt.Y('score:Q', scale=y_scale),
    text='label:N'
)

charts = []
for param in PARAMS:
    sub = agg[agg['parameter'] == param]
    band = alt.Chart(sub).mark_errorband().encode(
        x='value:Q',
        y=alt.Y('mean:Q', scale=y_scale),
        yError='ci95:Q',
        color=alt.Color('resolution:N', title='Base Resolution', scale=color_scale, sort=res_order)    )
    line = alt.Chart(sub).mark_line(point=True).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', title='Mean CGVQM Score', scale=y_scale),
        color=alt.Color('resolution:N', title='Base Resolution', scale=color_scale, sort=res_order),
        tooltip=['resolution', 'value', alt.Tooltip('mean:Q', format='.3f'), alt.Tooltip('ci95:Q', format='.3f')]
    )
    charts.append((band + line).properties(title=f'Parameter: {param}', width=300, height=220))

alt.vconcat(*[alt.hconcat(*charts[:2]), alt.hconcat(*charts[2:])]).resolve_scale(color='shared')

alt.VConcatChart(...)

#### Trying Friedman Test 

In [106]:
agg = df.groupby(['parameter', 'resolution', 'value'])['score_centered'].agg(['mean', 'std', 'count']).reset_index()
agg['ci95'] = 1.96 * (agg['std'] / np.sqrt(agg['count']))
agg['resolution'] = agg['resolution'].astype(str) + '%'

# Compute global y domain
y_min = (agg['mean'] - agg['ci95']).min()
y_max = (agg['mean'] + agg['ci95']).max()
y_scale = alt.Scale(domain=[y_min, y_max])

charts = []
for param in PARAMS:
    sub = agg[agg['parameter'] == param]
    band = alt.Chart(sub).mark_errorband(opacity=0.2).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', title='Mean Centered CGVQM', scale=y_scale),
        yError='ci95:Q',
        color=alt.Color('resolution:N', title='Base Resolution', scale=color_scale, sort=res_order)
    )
    line = alt.Chart(sub).mark_line(point=True).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', scale=y_scale),
        color=alt.Color('resolution:N', title='Base Resolution', scale=color_scale, sort=res_order),
        tooltip=['resolution', 'value', alt.Tooltip('mean:Q', format='.3f'), alt.Tooltip('ci95:Q', format='.3f')]
    )
    charts.append((band + line).properties(title=f'{param}', width=280, height=200))

alt.hconcat(*charts).resolve_scale(color='shared', x='independent').properties(
    title='TAA Parameter Response Curves (mean-centered CGVQM)'
)

alt.HConcatChart(...)

In [119]:
from scipy.stats import friedmanchisquare

# --- Mean range per (parameter x resolution) ---
effect_records = []

for (param, res), group in df.groupby(['parameter', 'resolution']):
    scene_ranges = group.groupby('scene').apply(
        lambda s: s.groupby('value')['score'].mean().max() - s.groupby('value')['score'].mean().min()
    )
    effect_records.append({
        'parameter': param,
        'resolution': res,
        'mean_range': scene_ranges.mean(),
        'se_range': scene_ranges.sem()
    })

effect_df = pd.DataFrame(effect_records)

# --- Friedman p-values per (parameter x resolution) ---
friedman_records = []

for (param, res), group in df.groupby(['parameter', 'resolution']):
    value_counts = group.groupby('scene')['value'].nunique()
    n_values = group['value'].nunique()
    complete_scenes = value_counts[value_counts == n_values].index
    group = group[group['scene'].isin(complete_scenes)]
    if group['scene'].nunique() < 3:
        continue
    values = sorted(group['value'].unique())
    groups_by_value = [group[group['value'] == v]['score_centered'].values for v in values]
    if any(len(g) < 3 for g in groups_by_value):
        continue
    _, friedman_p = friedmanchisquare(*groups_by_value)
    friedman_records.append({'parameter': param, 'resolution': res, 'friedman_p': friedman_p})

friedman_df = pd.DataFrame(friedman_records)

# --- Merge ---
summary_df = effect_df.merge(friedman_df, on=['parameter', 'resolution'])
summary_df['significant'] = summary_df['friedman_p'] < 0.05
summary_df['range_label'] = summary_df['mean_range'].apply(lambda x: f'{x:.2f}')

# --- Plot ---
base = alt.Chart(summary_df)

heatmap = base.mark_rect().encode(
    x=alt.X('resolution:O', title='Base Resolution', sort=[50, 71, 87, 100]),
    y=alt.Y('parameter:N', title='TAA Parameter'),
    color=alt.Color('mean_range:Q',
                    scale=alt.Scale(scheme='blues', domain=[0, 5]),
                    title='Mean Score Range (pts)'),
    tooltip=['parameter', 'resolution',
             alt.Tooltip('mean_range:Q', format='.3f'),
             alt.Tooltip('friedman_p:Q', format='.4f'),
             'significant']
)

# Switch text color based on value so it's readable on dark cells
range_text = base.mark_text(fontSize=11, dy=-6).encode(
    x=alt.X('resolution:O', sort=[50, 71, 87, 100]),
    y=alt.Y('parameter:N'),
    text='range_label:N',
    color=alt.condition(
        alt.datum.mean_range > 4.5,
        alt.value('white'),
        alt.value('black')
    )
)

sig_text = base.mark_text(fontSize=14, dy=6).encode(
    x=alt.X('resolution:O', sort=[50, 71, 87, 100]),
    y=alt.Y('parameter:N'),
    text=alt.condition(alt.datum.significant, alt.value('✱'), alt.value('')),
    color=alt.condition(
        alt.datum.mean_range > 4.5,
        alt.value('white'),
        alt.value('black')
    )
)

(heatmap + range_text + sig_text).properties(
    width=300, height=200,
    title='TAA Parameter Effect (Mean Score Range) × Base Resolution'
)

/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_43257/518858995.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scene_ranges = group.groupby('scene').apply(
/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_43257/518858995.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scene_ranges = group.groupby('scene').apply(
/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_43257/518858995.p

alt.LayerChart(...)

In [116]:
eta_records = []

for (param, res), group in df.groupby(['parameter', 'resolution']):
    grand_mean = group['score_centered'].mean()
    ss_between = sum(
        len(g) * (g['score_centered'].mean() - grand_mean) ** 2
        for _, g in group.groupby('value')
    )
    ss_total = ((group['score_centered'] - grand_mean) ** 2).sum()
    eta2 = ss_between / ss_total if ss_total > 0 else 0
    eta_records.append({
        'parameter': param,
        'resolution': res,
        'eta_squared': round(eta2, 4)
    })

eta_df = pd.DataFrame(eta_records)

alt.Chart(eta_df).mark_bar().encode(
    x=alt.X('eta_squared:Q', title='η² (Effect Size)', scale=alt.Scale(domain=[0, eta_df['eta_squared'].max() * 1.1])),
    y=alt.Y('resolution:O', title='Base Resolution'),
    color=alt.Color('resolution:O', title='Resolution', legend=None),
    row=alt.Row('parameter:N', title='TAA Parameter'),
    tooltip=['parameter', 'resolution', alt.Tooltip('eta_squared:Q', format='.4f')]
).properties(
    width=400, height=60, title='Effect Size (η²) per Parameter × Resolution'
).resolve_scale(x='shared')

alt.Chart(...)

In [91]:
# Friedman p-values per (parameter x resolution)
friedman_records = []

for (param, res), group in df.groupby(['parameter', 'resolution']):
    value_counts = group.groupby('scene')['value'].nunique()
    n_values = group['value'].nunique()
    complete_scenes = value_counts[value_counts == n_values].index
    group = group[group['scene'].isin(complete_scenes)]
    if group['scene'].nunique() < 3:
        continue
    values = sorted(group['value'].unique())
    groups_by_value = [group[group['value'] == v]['score_centered'].values for v in values]
    if any(len(g) < 3 for g in groups_by_value):
        continue
    _, friedman_p = friedmanchisquare(*groups_by_value)
    friedman_records.append({'parameter': param, 'resolution': res, 'friedman_p': friedman_p})

friedman_df = pd.DataFrame(friedman_records)

range_df = df.groupby(['parameter', 'resolution', 'value'])['score_centered'] \
             .mean().reset_index() \
             .groupby(['parameter', 'resolution'])['score_centered'] \
             .agg(lambda x: x.max() - x.min()) \
             .reset_index() \
             .rename(columns={'score_centered': 'score_range'})

summary_df = eta_df.copy()
summary_df = summary_df.merge(friedman_df, on=['parameter', 'resolution'])
summary_df = summary_df.merge(range_df, on=['parameter', 'resolution'])
summary_df['significant'] = (summary_df['friedman_p'] < 0.05) & (summary_df['score_range'] >= 5)
summary_df['eta_label'] = summary_df['eta_squared'].apply(lambda x: f'{x:.3f}')

base = alt.Chart(summary_df)

heatmap = base.mark_rect().encode(
    x=alt.X('resolution:O', title='Base Resolution'),
    y=alt.Y('parameter:N', title='TAA Parameter'),
    color=alt.Color('eta_squared:Q',
                    scale=alt.Scale(scheme='yellowgreenblue', domain=[0, summary_df['eta_squared'].max()]),
                    title='η² (Effect Size)'),
    tooltip=['parameter', 'resolution',
             alt.Tooltip('eta_squared:Q', format='.4f'),
             alt.Tooltip('friedman_p:Q', format='.4f'),
             'significant']
)

eta_text = base.mark_text(fontSize=11, dy=-6).encode(
    x=alt.X('resolution:O'),
    y=alt.Y('parameter:N'),
    text='eta_label:N'
)

sig_text = base.mark_text(fontSize=14, dy=6, color='black').encode(
    x=alt.X('resolution:O'),
    y=alt.Y('parameter:N'),
    text=alt.condition(alt.datum.significant, alt.value('✱'), alt.value(''))
)

(heatmap + eta_text + sig_text).properties(
    width=300, height=200,
    title='TAA Parameter Effect Size (η²) × Base Resolution'
)

alt.LayerChart(...)

In [79]:
range_df.sort_values('score_range', ascending=False)

,parameter,resolution,score_range
0,alpha_weight,50,6.228206
1,alpha_weight,71,2.985974
3,alpha_weight,100,2.545201
2,alpha_weight,87,2.404934
10,hist_percent,87,1.441330
11,hist_percent,100,1.384776
9,hist_percent,71,1.241018
8,hist_percent,50,0.591127
15,num_samples,100,0.243258
7,filter_size,100,0.134524
